# WaveletV3AELG Successive-Halving Analysis

Loads per-dataset WaveletV3AELG search results for both TrendAELG and TrendAE trend blocks,
then compares them head-to-head and emits a markdown report.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)

ROOT = Path('experiments')
if not ROOT.exists():
    ROOT = Path('.')

RESULTS = ROOT / 'results'
REPORT_PATH = ROOT / 'analysis_reports' / 'wavelet_v3aelg_study_analysis.md'

DATASET_ORDER = ['m4', 'tourism', 'traffic', 'weather']
DATASET_LABELS = {
    'm4': 'M4-Yearly',
    'tourism': 'Tourism-Yearly',
    'traffic': 'Traffic-96',
    'weather': 'Weather-96',
}

# Configurable result paths — extend as new variants/datasets are added
STUDY_VARIANTS = {
    'trendaelg': {
        'label': 'TrendAELG',
        'search_csvs': {ds: RESULTS / ds / 'wavelet_v3aelg_trendaelg_study_results.csv' for ds in DATASET_ORDER},
        'cross_csv': RESULTS / 'wavelet_v3aelg_trendaelg_cross_dataset_results.csv',
    },
    'trendae': {
        'label': 'TrendAE',
        'search_csvs': {ds: RESULTS / ds / 'wavelet_v3aelg_trendae_study_results.csv' for ds in DATASET_ORDER},
        'cross_csv': RESULTS / 'wavelet_v3aelg_trendae_cross_dataset_results.csv',
    },
}

In [ ]:
def load_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for c in ['search_round', 'best_val_loss', 'trend_thetas_dim_cfg', 'latent_dim_cfg', 'basis_dim']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

# Load all data
all_data = {}
for vkey, vinfo in STUDY_VARIANTS.items():
    search_dfs = {ds: load_csv(path) for ds, path in vinfo['search_csvs'].items()}
    cross_df = load_csv(vinfo['cross_csv'])
    all_data[vkey] = {'search_dfs': search_dfs, 'cross_df': cross_df, 'label': vinfo['label']}

    print(f'--- {vinfo["label"]} ---')
    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        if df is None:
            print(f'  {ds}: missing')
        else:
            print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={sorted(df["search_round"].dropna().astype(int).unique().tolist())}')
    print(f'  cross rows: {0 if cross_df is None else len(cross_df)}')

In [ ]:
report = []
report.append('# WaveletV3AELG Study Analysis')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    search_dfs = vdata['search_dfs']
    report.append(f'# Trend Block: {label}')
    report.append('')

    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        report.append(f'## {label} — Dataset: {DATASET_LABELS[ds]}')
        report.append('')
        if df is None or df.empty:
            report.append(f'- Missing CSV: `{STUDY_VARIANTS[vkey]["search_csvs"][ds]}`')
            report.append('')
            continue

        report.append(f'- Rows: {len(df)}')
        report.append(f'- Unique configs: {df["config_name"].nunique()}')
        report.append('')

        # Successive-halving funnel
        report.append('### Successive-Halving Funnel')
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        funnel_rows = []
        prev = None
        for r in rounds:
            rdf = df[df['search_round'].astype('Int64') == r]
            n_cfg = rdf['config_name'].nunique()
            keep = '-' if prev is None else f'{n_cfg}/{prev} ({n_cfg/max(prev,1):.0%})'
            best = pd.to_numeric(rdf['best_val_loss'], errors='coerce').median()
            funnel_rows.append({'Round': r, 'Configs': n_cfg, 'Rows': len(rdf), 'Median best_val_loss': best, 'Kept': keep})
            prev = n_cfg
        report.append(pd.DataFrame(funnel_rows).to_markdown(index=False, floatfmt='.4f'))
        report.append('')

        # Round 3 leaderboard
        report.append('### Round 3 Leaderboard (Top 10)')
        r3 = df[df['search_round'].astype('Int64') == 3].copy()
        if r3.empty:
            report.append('Round 3 not available.')
            report.append('')
        else:
            grp = (
                r3.groupby(['config_name', 'wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg'], dropna=False)
                .agg(median_best_val_loss=('best_val_loss', 'median'), std_best_val_loss=('best_val_loss', 'std'), n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .head(10)
                .reset_index()
            )
            report.append(grp.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

        # Round 1 marginals
        report.append('### Round 1 Hyperparameter Marginals (median best_val_loss)')
        r1 = df[df['search_round'].astype('Int64') == 1].copy()
        for col in ['wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg']:
            if col not in r1.columns:
                continue
            mg = (
                r1.groupby(col)
                .agg(median_best_val_loss=('best_val_loss', 'median'), mean_best_val_loss=('best_val_loss', 'mean'), n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .reset_index()
            )
            report.append(f'#### {col}')
            report.append(mg.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

In [ ]:
# Trend block comparison: TrendAELG vs TrendAE head-to-head
report.append('# Trend Block Comparison: TrendAELG vs TrendAE')
report.append('')

for ds in DATASET_ORDER:
    dfs_by_variant = {}
    for vkey, vdata in all_data.items():
        df = vdata['search_dfs'][ds]
        if df is not None and not df.empty:
            r3 = df[df['search_round'].astype('Int64') == 3].copy()
            if not r3.empty:
                dfs_by_variant[vkey] = r3

    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')
    if len(dfs_by_variant) < 2:
        report.append('Not enough data for comparison (need both TrendAELG and TrendAE round-3 results).')
        report.append('')
        continue

    # Compare by wavelet family
    report.append('### By Wavelet Family')
    compare_rows = []
    for vkey, r3 in dfs_by_variant.items():
        label = all_data[vkey]['label']
        by_wf = (
            r3.groupby('wavelet_family')
            .agg(median_bvl=('best_val_loss', 'median'))
            .reset_index()
        )
        by_wf['trend_block'] = label
        compare_rows.append(by_wf)

    cmp = pd.concat(compare_rows, ignore_index=True)
    pivot = cmp.pivot(index='wavelet_family', columns='trend_block', values='median_bvl')
    if 'TrendAELG' in pivot.columns and 'TrendAE' in pivot.columns:
        pivot['diff'] = pivot['TrendAELG'] - pivot['TrendAE']
        pivot = pivot.sort_values('diff')
    report.append(pivot.to_markdown(floatfmt='.4f'))
    report.append('')

    # Overall
    for vkey, r3 in dfs_by_variant.items():
        label = all_data[vkey]['label']
        report.append(f'- **{label}** overall median best_val_loss: {r3["best_val_loss"].median():.4f}')
    report.append('')

In [ ]:
# Cross-dataset robustness
report.append('# Cross-Dataset Robustness')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    cross_df = vdata['cross_df']
    report.append(f'## {label}')
    report.append('')
    if cross_df is None or cross_df.empty:
        report.append(f'- Missing or empty cross CSV: `{STUDY_VARIANTS[vkey]["cross_csv"]}`')
        report.append('')
        continue
    cdf = cross_df.copy()
    metric = 'best_val_loss'
    if metric not in cdf.columns and 'final_val_loss' in cdf.columns:
        metric = 'final_val_loss'
    board = (
        cdf.groupby(['canonical_config_id', 'source_datasets'], dropna=False)
        .agg(mean_metric=(metric, 'mean'), std_metric=(metric, 'std'), n=(metric, 'count'))
        .sort_values('mean_metric')
        .reset_index()
    )
    report.append(board.to_markdown(index=False, floatfmt='.4f'))
    report.append('')
    report.append(f'- Total cross rows: {len(cdf)}')
    report.append('')

In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print(f'Wrote report to: {REPORT_PATH}')
print('\n'.join(report[:30]))